# 01 — Disease-Enriched Subcluster Detection

Detect spatially contiguous UMAP regions enriched for disease cells using
`scutils.tl.detect_disease_enriched_subclusters`.

**Workflow:**
1. Load the AnnData object (must have kNN graph + UMAP pre-computed)
2. **Configure** detection parameters ← edit the configuration cell
3. Derive diseases to test (all categories except `CONTROL_CATEGORIES`)
4. Run disease-enriched subcluster detection
5. Visualise results on UMAP
6. Inspect subcluster statistics
7. Save outputs for downstream analysis (notebooks 02–05)

**Multi-dataset usage:** Run this notebook once per dataset with a unique
`DATASET_LABEL`. Results are saved to `RESULTS_DIR / DATASET_LABEL /`.
Downstream notebooks (03–05) aggregate across multiple dataset directories.

## 1. Import Libraries

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

import scutils
from scutils.tools.disease_subclusters import (
    detect_disease_enriched_subclusters,
    plot_disease_enriched_subclusters,
)

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 1

print(f"scutils  {scutils.__version__}")
print(f"scanpy   {sc.__version__}")

## 2. Configuration

**Edit the cell below** before running the rest of the notebook.

Key parameters:
- `ADATA_PATH` — path to your `.h5ad` file (must have kNN graph + UMAP)
- `DATASET_LABEL` — unique identifier for this dataset run
- `DISEASE_KEY` / `CELLTYPE_KEY` — column names in `adata.obs`
- `CONTROL_CATEGORIES` — conditions to exclude from disease testing and use as reference
- `COMBINE_DISEASES` — `"pool"` (default), `"union"`, or `None`
- `DISEASES_TO_TEST` — `None` = all non-control; or a list of specific diseases

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Input / Output ────────────────────────────────────────────────────────────
ADATA_PATH    = Path("path/to/adata.h5ad")
DATASET_LABEL = "dataset_A"           # unique label for this dataset run
RESULTS_DIR   = Path("results/disease_subclusters")

# ── Column names in adata.obs ─────────────────────────────────────────────────
DISEASE_KEY   = "disease"             # disease / condition column
CELLTYPE_KEY  = "cell_type"           # cell-type annotation column

# ── Control / reference categories ────────────────────────────────────────────
# Categories in adata.obs[DISEASE_KEY] that represent healthy / control.
# These are excluded from disease testing and used as the reference group.
CONTROL_CATEGORIES: list[str] = ["Control"]

# ── Disease selection ─────────────────────────────────────────────────────────
# Set to None to test ALL diseases except CONTROL_CATEGORIES.
# Set to a list (e.g. ["AD"]) to test only specific diseases.
DISEASES_TO_TEST: list[str] | None = None

# ── Detection mode ────────────────────────────────────────────────────────────
# "pool"  — merge all test diseases into one binary label (default)
# "union" — screen per-disease, take union of enriched micro-clusters
# None    — independent per-disease pipeline
COMBINE_DISEASES: str | None = "pool"

# ── Detection parameters ──────────────────────────────────────────────────────
MIN_ENRICHMENT_FOLD  = 2.0
MIN_SUBCLUSTER_SIZE  = 100
ENRICHMENT_FDR       = 0.05
SPATIAL_SENSITIVITY  = "medium"   # "low", "medium", "high", "very_high"
GMM_MAX_COMPONENTS   = 5
GMM_COVARIANCE_TYPE  = "full"
MIN_REFERENCE_CELLS  = 50

# ── Output key ────────────────────────────────────────────────────────────────
RESULT_KEY = "disease_subcluster"

# ── Plotting ──────────────────────────────────────────────────────────────────
NCOLS         = 3
FIGSIZE_PANEL = (5, 5)
SAVE_FIGS     = True

print("Configuration loaded.")
print(f"  Dataset label       : {DATASET_LABEL}")
print(f"  AnnData path        : {ADATA_PATH}")
print(f"  Disease key         : {DISEASE_KEY}")
print(f"  Cell-type key       : {CELLTYPE_KEY}")
print(f"  Control categories  : {CONTROL_CATEGORIES}")
print(f"  Diseases to test    : {DISEASES_TO_TEST or 'all non-control'}")
print(f"  Combine diseases    : {COMBINE_DISEASES!r}")
print(f"  Spatial sensitivity : {SPATIAL_SENSITIVITY}")
print(f"  Results dir         : {RESULTS_DIR / DATASET_LABEL}")

## 3. Load Data

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print(adata)

# Quick summary of disease and cell-type distributions
for col in [DISEASE_KEY, CELLTYPE_KEY]:
    if col in adata.obs.columns:
        counts = adata.obs[col].value_counts()
        print(f"\n{col} ({len(counts)} categories):\n{counts.to_string()}")

## 4. Derive Diseases to Test

If `DISEASES_TO_TEST` is `None`, automatically select all disease categories
in `adata.obs[DISEASE_KEY]` except those in `CONTROL_CATEGORIES`.

In [ ]:
all_categories = adata.obs[DISEASE_KEY].unique().tolist()
control_set = set(CONTROL_CATEGORIES)

# Validate that control categories exist
missing_ctrl = control_set - set(all_categories)
if missing_ctrl:
    print(
        f"⚠  CONTROL_CATEGORIES not found in adata.obs['{DISEASE_KEY}']: "
        f"{missing_ctrl}\n"
        f"   Available: {all_categories}"
    )

if DISEASES_TO_TEST is None:
    diseases_to_test = [c for c in all_categories if c not in control_set]
else:
    diseases_to_test = DISEASES_TO_TEST

reference_group = [c for c in CONTROL_CATEGORIES if c in set(all_categories)]
if not reference_group:
    reference_group = None
    print("⚠  No valid reference group — enrichment will use full dataset.")

print(f"Diseases to test : {diseases_to_test}")
print(f"Reference group  : {reference_group}")

## 5. Run Disease-Enriched Subcluster Detection

In [ ]:
detect_disease_enriched_subclusters(
    adata,
    disease_key=DISEASE_KEY,
    celltype_key=CELLTYPE_KEY,
    groups_disease=diseases_to_test,
    reference_group=reference_group,
    combine_diseases=COMBINE_DISEASES,
    min_enrichment_fold=MIN_ENRICHMENT_FOLD,
    min_subcluster_size=MIN_SUBCLUSTER_SIZE,
    enrichment_fdr=ENRICHMENT_FDR,
    spatial_sensitivity=SPATIAL_SENSITIVITY,
    gmm_max_components=GMM_MAX_COMPONENTS,
    gmm_covariance_type=GMM_COVARIANCE_TYPE,
    min_reference_cells=MIN_REFERENCE_CELLS,
    result_key=RESULT_KEY,
    verbose=True,
)

# Summary
labels = adata.obs[RESULT_KEY]
n_enriched = (labels != "background").sum()
n_subclusters = labels[labels != "background"].nunique()
print(f"\n✓ Detection complete.")
print(f"  Enriched cells  : {n_enriched:,} / {adata.n_obs:,}")
print(f"  Subclusters     : {n_subclusters}")

## 6. Visualise Results

UMAP panels split by cell type, showing disease-enriched subclusters.

In [ ]:
fig = plot_disease_enriched_subclusters(
    adata,
    celltype_key=CELLTYPE_KEY,
    disease_key=DISEASE_KEY,
    result_key=RESULT_KEY,
    split_by="celltype",
    ncols=NCOLS,
    figsize_panel=FIGSIZE_PANEL,
    show=False,
)

output_dir = RESULTS_DIR / DATASET_LABEL
if SAVE_FIGS:
    output_dir.mkdir(parents=True, exist_ok=True)
    fig_path = output_dir / "subclusters_by_celltype.png"
    fig.savefig(fig_path, bbox_inches="tight")
    print(f"Saved: {fig_path}")

plt.show()
plt.close(fig)

In [ ]:
fig = plot_disease_enriched_subclusters(
    adata,
    celltype_key=CELLTYPE_KEY,
    disease_key=DISEASE_KEY,
    result_key=RESULT_KEY,
    split_by="disease",
    ncols=NCOLS,
    figsize_panel=FIGSIZE_PANEL,
    show=False,
)

if SAVE_FIGS:
    fig_path = output_dir / "subclusters_by_disease.png"
    fig.savefig(fig_path, bbox_inches="tight")
    print(f"Saved: {fig_path}")

plt.show()
plt.close(fig)

## 7. Inspect Subcluster Statistics

In [ ]:
info_key = f"{RESULT_KEY}_info"
info_df = adata.uns[info_key].copy()

print(f"Subcluster info: {len(info_df)} entries")

display(
    info_df.style
    .format({
        "fold_enrichment": "{:.2f}",
        "pvalue":          "{:.2e}",
        "pvalue_adj":      "{:.2e}",
    })
    .background_gradient(subset=["fold_enrichment"], cmap="Reds")
)

## 8. Save Results

Save the annotated AnnData and the subcluster info table for use in
downstream notebooks (02–05).

In [ ]:
output_dir = RESULTS_DIR / DATASET_LABEL
output_dir.mkdir(parents=True, exist_ok=True)

# Save AnnData with subcluster labels
adata_path = output_dir / "adata_with_subclusters.h5ad"
adata.write_h5ad(adata_path)
print(f"Saved AnnData: {adata_path}")

# Save subcluster info table
info_path = output_dir / "subcluster_info.csv"
info_df.to_csv(info_path, index=False)
print(f"Saved info table: {info_path}")

print(f"\n✓ All outputs saved to {output_dir}")

---

## Summary

| Step | Function | Key parameters |
|------|----------|----------------|
| Load data | `sc.read_h5ad()` | `ADATA_PATH` |
| Detect subclusters | `detect_disease_enriched_subclusters()` | `COMBINE_DISEASES`, `SPATIAL_SENSITIVITY`, `MIN_ENRICHMENT_FOLD` |
| Visualise | `plot_disease_enriched_subclusters()` | `split_by`, `NCOLS`, `FIGSIZE_PANEL` |
| Save | `adata.write_h5ad()`, `pd.to_csv()` | `RESULTS_DIR / DATASET_LABEL` |

**Next:** Run `02_differential_expression.ipynb` with the same `DATASET_LABEL`
to perform DE on each detected subcluster.